# 04. Low-light Enhancement - zero_dce

| 항목 | 내용 |
|---|---|
| 대상 모델 | `zero_dce.dxnn` |
| 카테고리 | **Low-light Enhancement** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 정규화된 `float32` CHW 입력을 만드는 전처리를 `uint8` 입력 모델과 비교해 설명할 수 있다.
2. 밝기 향상 곡선 `E + α·E·(1-E)` 를 값 하나에 직접 적용해 계산할 수 있다.
3. Task 세 개의 흐름을 보고 어느 계산이 모델 안에서 끝났는지 판단할 수 있다.
4. 밝아진 정도를 평균·분위수로 측정해 제시할 수 있다.

## 1. Low-light Enhancement (저조도 향상)

**어둡게 찍힌 영상을 받아 픽셀마다 밝기를 끌어올려 보이게 만드는 작업**

어두운 사진은 값이 0 근처에 몰려 있어 눈으로도 화면으로도 구분되지 않는다. 이 작업은 그 좁은 구간을 넓게
펼쳐 형태가 드러나게 한다.

**출력의 성격**

픽셀마다 값이 하나씩 나오는 영상이며 크기는 입력과 같다. 어둠에 묻혀 아예 기록되지 않은 부분은 되살릴 수
없고, 값이 남아 있는 만큼만 펼쳐진다. 노이즈도 함께 밝아지므로 어두운 곳의 잡티가 더 눈에 띈다.

**활용 사례**

- 야간 감시 영상의 시인성 개선
- 실내·역광 촬영 사진 보정
- 검출·인식 모델 앞단의 입력 품질 보정

**포함되지 않는 작업**

- 잡티 제거 (Denoising)
- 해상도 확대 (Super Resolution)
- 어두운 곳에 무엇이 있는지 판정 (Object Detection)

## 2. 전체 파이프라인

```text
sample_lowlight.jpg  600×400 BGR (밝기 평균 14.3)
  → 전처리         BGR→RGB · 600×400 resize · ÷255 · HWC→CHW float32
  → run()          입력 Tensor 하나를 전달
  → cpu_0          42   [1, 3, 400, 600]   (전처리 결과를 NPU 형식으로)
  → npu_0          45 · 16 · 28            (내부 전달, Public 아님)
  → cpu_1          87   [1, 3, 400, 600]   (Public Output — 밝아진 영상)
  → 후처리         clip(0,1) → CHW→HWC → RGB→BGR → 원본 크기 → ×255 → uint8
  → Result         EnhancedImageResult (밝아진 영상 한 장)
  → 시각화         원본과 결과를 좌우로 붙여 비교
```

![04. Low-light Enhancement 전체 파이프라인](resources/04_Low-light_Enhancement_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input.1` |
| Shape / Dtype | `[1, 3, 400, 600]` / `float32` |
| 채널 순서 | NCHW |
| Color / 값 범위 | RGB / 0.0~1.0 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | `÷255` 로 0~1 로 만듦 |

**앞의 모델들과 다른 점**

- **`float32` 입력** — 01~03번은 `uint8` 을 그대로 넣었지만 이 모델은 전처리가 0~1 로 나눈다. 밝기 향상
  곡선이 0~1 구간에서 정의되기 때문이다
- **NCHW 입력** — 채널 축이 앞에 온다. `cv2` 가 만든 배열은 HWC 이므로 축을 옮기는 단계가 필요하다
- **입력 크기가 4:6 비율** — 정사각이 아니며 sample 과 크기가 같아 이번 실행에서는 늘어남이 없다

### 3.2 출력

이 모델은 `.dxnn` 하나 안에 Task 가 셋이다. `run()` 이 돌려주는 값은 **마지막 CPU Task 가 만든 값**이다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| (모델 입력) | `input.1` | `[1, 3, 400, 600]` / `float32` | `cpu_0` · `npu_0` | 입력 |
| `cpu_0` | `42` | `[1, 3, 400, 600]` / `float32` | `npu_0` | 아님 — 내부 전달 |
| `npu_0` | `45` | `[1, 3, 400, 600]` / `float32` | `cpu_1` | 아님 — 내부 전달 |
| `npu_0` | `16` · `28` | `[1, 32, 400, 600]` / `float32` | `cpu_1` | 아님 — 내부 전달 |
| `cpu_1` | `87` | `[1, 3, 400, 600]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 32채널 중간 값은 밖으로 나오지
않으므로 노트북에서 볼 수 없고, `87` 하나만 손에 들어온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 3 | 채널. RGB 세 장 |
| 2 | 400 | 높이 |
| 3 | 600 | 너비 |

`float32` 하나가 4바이트이므로 `3 × 400 × 600` 은 2,880,000 바이트다.

**Zero-DCE 는 밝아진 영상을 직접 그리지 않고 밝기 곡선을 예측하는 구조다.** 픽셀 값 `E` 를 아래 식으로
여러 번 밀어 올리며, `α` 는 픽셀마다·채널마다 다른 값으로 모델이 예측한다.

```text
E ← E + α · E · (1 - E)      이 갱신을 8회 반복
```

식의 모양이 이렇게 정해진 이유는 두 끝을 건드리지 않기 위함이다. `E = 0` 이면 `α` 가 무엇이든 0 이고
`E = 1` 이면 변화량이 0 이라 1 을 넘지 않는다. 어두운 중간값만 크게 밀려 올라가므로 밝은 곳이 타지 않는다.

**곡선 반복이 이 배포본에서는 모델 안에서 끝난다.** `87` 은 곡선 파라미터가 아니라 이미 밝아진 3채널
영상이며, 채널이 3개인지 24개인지로 그 사실을 갈라낼 수 있다 — 24개면 8회분 `α`(8 × 3) 이므로 Host 가
직접 반복해야 한다. 이번 출력은 3채널이라 Host 후처리는 형식 변환만 한다.

#### 출력을 사람이 읽는 값으로 바꾸기

출력은 0~1 부근의 `float32` CHW RGB 이고 화면에 그릴 영상은 0~255 `uint8` HWC BGR 이다. 되돌리는 순서는
아래와 같다.

| 순서 | 하는 일 | 이유 |
|---|---|---|
| 1 | `clip(0, 1)` | 범위를 벗어난 값이 되돌릴 때 넘치지 않게 한다 |
| 2 | CHW → HWC | `cv2` 와 matplotlib 이 읽는 축 순서로 맞춘다 |
| 3 | RGB → BGR | 전처리에서 뒤집은 색 순서를 되돌린다 |
| 4 | 원본 크기로 resize | 모델 입력 크기와 원본이 다를 때 맞춘다 |
| 5 | `×255` 후 `uint8` | 0~1 실수를 0~255 정수로 바꾼다 |

이번 실행의 값은 최소 `0.0087`, 최대 `0.8153` 이라 `clip` 이 잘라 내는 픽셀이 없다. 예를 들어 값 `0.4646`
은 `0.4646 × 255 = 118.5` 이므로 `uint8` 118 이 된다.

곡선을 손으로 따라가 보면 값이 어떻게 올라가는지 보인다. 어두운 픽셀 `E = 0.10` 에 `α = 0.5` 를 한 번
적용하면 `0.10 + 0.5 × 0.10 × 0.90 = 0.145` 로 45% 밝아진다. 같은 `α` 로 `E = 0.90` 을 밀면
`0.90 + 0.5 × 0.90 × 0.10 = 0.945` 로 5% 만 오른다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. Task 가
`cpu_0` → `npu_0` → `cpu_1` 로 이어지고 `87` 만 model output 인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "zero_dce.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정한다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_lowlight.jpg"

print("input image:", IMAGE_PATH.name)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 크기를 담아 두어
  후처리가 결과를 그 크기로 되돌린다
- `EnhancedImageResult` — 후처리가 만들어 시각화로 넘기는 결과. 밝아진 영상 한 장을 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0
    normalized_input: np.ndarray = field(default=None)


@dataclass
class EnhancedImageResult:
    """밝아진 영상 한 장. 원본 대응: common/processors/zero_dce_postprocessor.py"""

    output_image: np.ndarray = field(default=None)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 입력이 NCHW `float32` 인 것과 출력이
`87` 하나인 것을 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][2], input_info[0]["shape"][3]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 RGB 로 뒤집고 600×400 으로 맞춘 뒤 255 로 나누고 축을 HWC 에서 CHW 로 옮긴다. 앞 세 노트북과
달리 **정규화와 축 이동이 전처리의 일**이며, 이 단계를 빼먹으면 모델이 값을 0~1 로 가정하고 계산하므로
결과가 통째로 어긋난다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 600×400
- dtype·범위 — `float32` 0.0~1.0
- 축 구성 — HWC → CHW
- 메모리 연속성 — 축을 옮기면 연속이 아니게 되므로 되돌려 준다

#### 입력 이미지 확인

원본 사진을 읽어 크기와 밝기 분포를 확인한다. 밝기 평균이 매우 낮아 화면에서는 거의 검게 보인다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
print(f"밝기  min={image.min()} max={image.max()} mean={image.mean():.2f}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `zero_dce_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `zero_dce_preprocessor_process` — RGB 변환·resize·정규화·축 이동을 거쳐 입력 Tensor 와 Context 로 변환

In [ ]:
def zero_dce_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def zero_dce_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    # 0~1 로 나눈 뒤 채널 축을 앞으로 옮긴다. transpose 는 뷰만 바꾸므로 연속 메모리로 되돌려
    # 런타임이 다시 복사하지 않게 한다.
    normalized = resized.astype(np.float32) / 255.0
    ctx.normalized_input = np.ascontiguousarray(np.transpose(normalized, (2, 0, 1)))

    return ctx.normalized_input, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 값의 최대가 1 에 한참 못 미치는 것이 사진이 어둡다는 뜻이며, 이 좁은
구간을 펼치는 것이 모델의 일이다.

In [ ]:
preprocessor_state = zero_dce_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = zero_dce_preprocessor_process(preprocessor_state, image)

notebook_describe_array("input_tensor", input_tensor)
print(f"dtype={input_tensor.dtype} contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 채널 수 — 3 이면 밝아진 영상, 24 면 곡선 파라미터
- 값 범위 — 0~1 부근이라 그대로 화면에 그릴 수 없다
- 입력과의 평균 비교 — 이미 밝아졌는지 여기서 드러난다

In [ ]:
raw = np.asarray(outputs[0])
notebook_describe_array("raw output", raw)

params = np.squeeze(raw)
print(f"squeeze 뒤 shape = {params.shape} → 채널 {params.shape[0]}개")
print("갈래 =", "밝아진 영상 (Host 는 형식 변환만)" if params.shape[0] == 3
      else f"곡선 파라미터 {params.shape[0] // 3}회분 (Host 가 반복 적용)")
print(f"평균  입력 tensor={input_tensor.mean():.4f} → 출력={params.mean():.4f}")

### 4.4 후처리

크기 1인 축을 눌러 `[3, 400, 600]` 으로 만들고 채널 수로 갈래를 정한 뒤 화면에 그릴 형식으로 되돌린다.
채널이 3개면 값을 그대로 쓰고, 24개면 정규화 입력에서 시작해 곡선을 8회 적용한 뒤 되돌린다.

**두 갈래를 모두 두는 이유는 같은 후처리가 두 종류의 배포본을 받기 때문이다.** 곡선 반복이 모델 안에
들어간 배포본과 파라미터만 내놓는 배포본이 함께 있어, 출력 채널 수로 어느 쪽인지 판정한다.

형식 변환 순서는 `clip(0,1)` → CHW→HWC → RGB→BGR → 원본 크기 → `×255` 이며 이 순서를 바꾸지 않는다.
색 순서를 되돌리지 않으면 오류 없이 색만 뒤집혀 나온다.

#### 후처리 함수 정의

- `zero_dce_postprocessor_create` — 곡선 반복 횟수를 state 에 담음. 기본값 8 은 후처리 코드의 값이다
- `zero_dce_apply_le_curve` — 곡선 갱신을 정해진 횟수만큼 적용
- `zero_dce_to_bgr_uint8` — CHW RGB 실수를 HWC BGR `uint8` 로 되돌림
- `zero_dce_postprocessor_process` — 채널 수로 갈래를 정해 Result 생성

In [ ]:
def zero_dce_postprocessor_create(num_iterations: int = 8) -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"num_iterations": num_iterations}


def zero_dce_apply_le_curve(enhanced: np.ndarray, params: np.ndarray, n_iters: int) -> np.ndarray:
    """E ← E + α·E·(1-E) 를 n_iters 회 적용한다.

    원본 대응: ZeroDCEPostprocessor._apply_le_curve()
    """
    for index in range(n_iters):
        alpha = np.clip(params[index * 3:(index + 1) * 3], 0.0, 1.0)
        enhanced = enhanced + alpha * enhanced * (1.0 - enhanced)
    return np.clip(enhanced, 0.0, 1.0)


def zero_dce_to_bgr_uint8(enhanced: np.ndarray, ctx: PreprocessContext) -> np.ndarray:
    """CHW RGB float 을 HWC BGR uint8 로 되돌린다. 원본 대응: ZeroDCEPostprocessor._to_bgr_uint8()"""
    bgr = np.ascontiguousarray(np.transpose(enhanced, (1, 2, 0))[:, :, ::-1])
    if bgr.shape[1] != ctx.original_width or bgr.shape[0] != ctx.original_height:
        bgr = cv2.resize(bgr, (ctx.original_width, ctx.original_height),
                         interpolation=cv2.INTER_LINEAR)
    return np.clip(bgr * 255.0, 0, 255).astype(np.uint8)


def zero_dce_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    params = np.squeeze(outputs[0])

    if params.shape[0] == 3:
        enhanced = np.clip(params, 0.0, 1.0)
    else:
        # 곡선 파라미터만 나온 배포본. 정규화 입력에서 시작해 직접 반복한다.
        enhanced = zero_dce_apply_le_curve(ctx.normalized_input.copy(), params,
                                           params.shape[0] // 3)

    return [EnhancedImageResult(output_image=zero_dce_to_bgr_uint8(enhanced, ctx))]

#### 후처리 실행

확인할 것:

- dtype·shape — `uint8` `[400, 600, 3]` 로 원본 크기
- 밝기 평균 — 몇 배 밝아졌는지
- 분위수 — 어두운 쪽이 더 크게 올라갔는지
- 표준편차 — 값이 넓게 퍼졌는지

In [ ]:
postprocessor_state = zero_dce_postprocessor_create()
results = zero_dce_postprocessor_process(postprocessor_state, outputs, ctx)
enhanced = results[0].output_image

print(f"enhanced      shape={enhanced.shape} dtype={enhanced.dtype} "
      f"min={enhanced.min()} max={enhanced.max()}")
print(f"밝기 평균     입력={image.mean():.2f} → 결과={enhanced.mean():.2f} "
      f"({enhanced.mean() / image.mean():.2f}배)")

gray_in = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray_out = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
print(f"표준편차      입력={gray_in.std():.2f} → 결과={gray_out.std():.2f}")
for percent in (5, 50, 95):
    print(f"  {percent:2d}분위     입력={np.percentile(gray_in, percent):6.1f} → "
          f"결과={np.percentile(gray_out, percent):6.1f}")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 원본과 결과를 좌우로 붙인다. 결과를 원본 크기로 맞춘 뒤 붙이므로 두 장의 높이가 같다.
그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def zero_dce_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본과 좌우로 붙여 그린다. 원본 대응: Runner.visualize()"""
    enhanced = results[0].output_image
    if enhanced.ndim == 2:
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

    height, width = image.shape[:2]
    canvas = np.hstack([image, cv2.resize(enhanced, (width, height))])

    cv2.putText(canvas, "Input", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.putText(canvas, "Enhanced", (width + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 0), 2)
    return canvas

#### 결과 시각화

좌우 비교를 inline 으로 표시한다. 왼쪽은 거의 검은 원본이고 오른쪽은 형태가 드러난 결과다.

In [ ]:
output_image = zero_dce_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Input vs Enhanced", figsize=(13, 5))

#### 밝기 분포가 어떻게 옮겨졌는지

밝기 히스토그램을 나란히 그리면 값이 몰려 있던 구간이 어디로 옮겨졌는지 보인다. 막대 하나가 밝기 한 칸의
픽셀 수이며, 왼쪽 끝에 뭉쳐 있던 분포가 가운데로 퍼진다.

In [ ]:
hist_in = cv2.calcHist([gray_in], [0], None, [64], [0, 256]).flatten()
hist_out = cv2.calcHist([gray_out], [0], None, [64], [0, 256]).flatten()

# 히스토그램을 막대 그림으로 직접 그린다. 글자는 영문만 쓴다.
chart = np.full((260, 560, 3), 255, dtype=np.uint8)
scale = 200.0 / max(hist_in.max(), hist_out.max())
for index in range(64):
    x = 30 + index * 8
    cv2.rectangle(chart, (x, 240), (x + 3, 240 - int(hist_in[index] * scale)), (60, 60, 220), -1)
    cv2.rectangle(chart, (x + 3, 240), (x + 6, 240 - int(hist_out[index] * scale)),
                  (60, 180, 60), -1)
cv2.putText(chart, "Input (red) vs Enhanced (green)", (30, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
            (30, 30, 30), 1, cv2.LINE_AA)
cv2.putText(chart, "0", (26, 256), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (90, 90, 90), 1, cv2.LINE_AA)
cv2.putText(chart, "255", (515, 256), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (90, 90, 90), 1, cv2.LINE_AA)

notebook_show_bgr(chart, "Brightness histogram", figsize=(9, 5))